# Train PhoBertSpoTagger trên Colab

Notebook này chỉ lo phần môi trường (clone code, gắn dữ liệu/model từ Drive, cài thư viện) —
logic train thật sự nằm trong repo (`src/extraction/phobert_spo_tagger.py`,
`scripts/spo_extraction/train_phobert_spo_tagger.py`), đồng bộ qua git.

**Trước khi chạy:**
1. Đổi `DRIVE_ROOT` bên dưới nếu bạn để dữ liệu ở thư mục Drive khác.
2. Đã upload sẵn `data/SPO/relations.csv` vào `DRIVE_ROOT/data/SPO/relations.csv` trên Drive
   (sinh file này bằng `python -m scripts.spo_extraction.build_spo_relations` từ
   `data/llm_relations/relations.csv`, không chỉnh sửa tay qua Excel/Sheets — dễ làm hỏng
   quoting/field CSV).
3. Chọn Runtime > Change runtime type > GPU trước khi chạy (nếu có GPU free trên Colab).


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. Clone / pull code từ git

Repo public, không cần token. Nếu đã clone từ lần trước, cell này sẽ `git pull` thay vì clone lại.

In [ ]:
REPO_URL = "https://github.com/thnghia-ctu/CausalGraph.git"
BRANCH = "v3"
REPO_DIR = "/content/CausalGraph"

import os

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    !git clone -b $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR


## 3. Gắn `data/` và `models/` vào Drive

Hai thư mục này bị `.gitignore`, không nằm trong git — clone xong sẽ trống hoặc không tồn tại.
Symlink sang Drive để dữ liệu và checkpoint được giữ lại qua các session, không cần copy tay
mỗi lần mở lại Colab.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/CausalGraph"

import os

os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)

!rm -rf {REPO_DIR}/data {REPO_DIR}/models
!ln -s {DRIVE_ROOT}/data {REPO_DIR}/data
!ln -s {DRIVE_ROOT}/models {REPO_DIR}/models

!ls -la {REPO_DIR}/data/SPO/ 2>/dev/null || echo "Chưa có data/SPO/relations.csv trên Drive — upload trước khi train."


## 4. Cài thư viện

In [ ]:
!pip install -q -r requirements.txt


## 5. (Tuỳ chọn) Đăng nhập Hugging Face Hub

Cần nếu muốn push model đã train lên Hub ở bước cuối. Token tạo tại
https://huggingface.co/settings/tokens (quyền write).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 6. Train

Checkpoint được lưu định kỳ vào `models/spo_tagger/` (= Drive, qua symlink ở bước 3).
Nếu Colab bị ngắt kết nối giữa chừng, chỉ cần chạy lại cell này — `PhoBertSpoTagger.fit`
tự resume từ checkpoint gần nhất thay vì train lại từ đầu.

In [ ]:
!python -m scripts.spo_extraction.train_phobert_spo_tagger


## 7. (Tuỳ chọn) Push model lên Hugging Face Hub

Lưu bản train xong lên Hub để có version history riêng cho model, tách khỏi git,
và tải lại được từ máy local hoặc lần train sau mà không cần train lại.

In [ ]:
from src.extraction.phobert_spo_tagger import PhoBertSpoTagger
from configs.config import SPO_TAGGER_MODEL_DIR

HF_REPO_ID = "your-username/vi-spo-tagger"  # đổi thành repo của bạn

tagger = PhoBertSpoTagger.load(SPO_TAGGER_MODEL_DIR / "final")
tagger.push_to_hub(HF_REPO_ID)
